# Advice model: QDoRA fine-tune of Gemma 4 12B

Trains the advice model on `train.jsonl` (made on the PC with
`python -m best_of_sno.advice export-train`). Design:
`docs/superpowers/specs/2026-11-03-advice-model-design.md`.

**Before you start**
1. Colab Pro, runtime **GPU: A100** (or L4) with **High-RAM**. The free T4 is too small for 12B at this length.
2. On the PC, run `python -m best_of_sno.advice check-train`, then upload
   `best_of_sno/data/advice/train.jsonl` to Google Drive at `MyDrive/sno-advisor/train.jsonl`.
   It holds full story text: keep that Drive folder private.
3. Run the cells in order. Training saves to Drive after every epoch, so if Colab
   disconnects, run everything again and it resumes.

**What comes back:** a LoRA/DoRA adapter in `MyDrive/sno-advisor/r0/adapter` (the real
result, ~100 MB) and, from the last section, a `Q4_K_M` GGUF to load into Ollama on the PC.

Steps 1–6 are the training. Steps 7–8 (merge, GGUF) are the riskiest: they depend on
llama.cpp supporting Gemma 4. If they fail, the adapter is still safe on Drive.

## 1. Settings

In [ ]:
BRANCH = "main"                          # repo branch with best_of_sno/advice_train.py
BASE_MODEL = "google/gemma-4-12B-it"     # must match the PC's gemma4:12b (Gemma 4, 11.9B)
DRIVE_DIR = "/content/drive/MyDrive/sno-advisor"
RUN = "r0"                               # round name; outputs go to DRIVE_DIR/RUN

# From the design doc
RANK, ALPHA, DROPOUT = 16, 32, 0.05
EPOCHS, LR = 2, 1e-4
BATCH, GRAD_ACCUM = 1, 8                 # 8 examples per update
MAX_LEN = 8192                           # the longest example is ~6.5k tokens

## 2. Install and check the GPU

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate
import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100 or L4"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.cuda.get_device_name(0), f"{gb:.0f} GB")
if gb < 22:
    print("WARNING: under 22 GB of GPU memory; 12B at 8k tokens will likely run out. Use an A100 or L4.")
assert torch.cuda.is_bf16_supported(), "This GPU has no bf16; use an A100 or L4"

## 3. Drive, the repo and the data

In [ ]:
import os, sys, json
from google.colab import drive
drive.mount("/content/drive")
OUT = f"{DRIVE_DIR}/{RUN}"
os.makedirs(OUT, exist_ok=True)

if not os.path.exists("/content/repo"):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/jasqnlu/Best-of-sno-llm-advisor /content/repo
sys.path.insert(0, "/content/repo")
from best_of_sno import advice_train as T   # the tested masking and split

records = T.load_jsonl(f"{DRIVE_DIR}/train.jsonl")
train_recs, val_recs = T.split(records)
print(len(records), "examples:", len(train_recs), "train,", len(val_recs), "validation")

## 4. Tokenizer and the prompt format

Checks that the chat template renders the same prompt Ollama sends with thinking off,
ending in an empty thought block, and that `<turn|>` is one token.

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(BASE_MODEL)
tok = processor.tokenizer
if not getattr(tok, "chat_template", None):
    tok.chat_template = processor.chat_template

sample = T.render_prompt(tok, "Hello")
print(repr(sample))
assert sample.rstrip().endswith("<channel|>"), "unexpected template: thinking-off prompt should end in an empty thought"
assert "<|turn>model" in sample
eot = tok(T.END_OF_TURN, add_special_tokens=False)["input_ids"]
assert len(eot) == 1, f"<turn|> is {len(eot)} tokens; check the template"

train_ex, too_long = T.build_all(tok, train_recs, MAX_LEN)
val_ex, too_long_val = T.build_all(tok, val_recs, MAX_LEN)
lens = sorted(len(e["input_ids"]) for e in train_ex)
print(f"train {len(train_ex)} (dropped {len(too_long)} too long), val {len(val_ex)} (dropped {len(too_long_val)})")
print("tokens: median", lens[len(lens) // 2], "max", lens[-1])
print("advice tokens per example:", sum(sum(l != T.IGNORE for l in e["labels"]) for e in train_ex) // len(train_ex))

## 5. Load Gemma in 4-bit and add DoRA adapters

In [ ]:
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                                 dtype=torch.bfloat16, device_map={"": 0})
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})

# Adapters on the language model's attention and MLP projections only, not the
# image/audio parts (training is text only).
PROJ = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
SKIP = ("vision", "audio", "image", "embed_", "multi_modal", "mm_")
targets = [n for n, m in model.named_modules()
           if n.split(".")[-1] in PROJ and not any(s in n.lower() for s in SKIP)]
print(len(targets), "projection layers, e.g.", targets[:3])
assert targets, "no projection layers matched; print [n for n, _ in model.named_modules()] and adjust PROJ/SKIP"

config = LoraConfig(r=RANK, lora_alpha=ALPHA, lora_dropout=DROPOUT, use_dora=True,
                    target_modules=targets, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, config)
model.print_trainable_parameters()

## 6. Train (saves to Drive every epoch; re-running resumes)

In [ ]:
import glob
from transformers import Trainer, TrainingArguments

class Examples(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
def collate(batch):
    return {k: torch.tensor(v) for k, v in T.pad_batch(batch, pad_id).items()}

args = TrainingArguments(
    output_dir=f"{OUT}/checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM, lr_scheduler_type="cosine", warmup_ratio=0.03,
    bf16=True, optim="paged_adamw_8bit", gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5, eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    report_to="none", remove_unused_columns=False, seed=T.SEED)
trainer = Trainer(model=model, args=args, train_dataset=Examples(train_ex),
                  eval_dataset=Examples(val_ex), data_collator=collate)
resume = bool(glob.glob(f"{OUT}/checkpoints/checkpoint-*"))
print("resuming" if resume else "starting fresh")
trainer.train(resume_from_checkpoint=resume or None)
print(trainer.evaluate())
model.save_pretrained(f"{OUT}/adapter")
print("adapter saved to", f"{OUT}/adapter")

### Look at one piece of advice from a validation story

In [ ]:
model.eval()
model.config.use_cache = True
rec = val_recs[0]
prompt_ids = tok(T.render_prompt(tok, rec["messages"][0]["content"]), add_special_tokens=False,
                 return_tensors="pt").input_ids.to(model.device)
with torch.no_grad():
    out = model.generate(prompt_ids, max_new_tokens=700, do_sample=False,
                         eos_token_id=tok.convert_tokens_to_ids(T.END_OF_TURN))
print("MODEL:\n" + tok.decode(out[0, prompt_ids.shape[1]:], skip_special_tokens=True))
print("\nCLAUDE (training example):\n" + rec["messages"][1]["content"])

## 7. Merge the adapter into the full model

Reloads Gemma in bf16 on the CPU (needs High-RAM, ~30 GB) and folds the adapter in.
If Colab restarts here, skip steps 1–6: run steps 1–3, then this cell.

In [ ]:
import gc, torch
from transformers import AutoModelForMultimodalLM, AutoProcessor
from peft import PeftModel
for name in ("model", "trainer"):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()

MERGED = "/content/merged"
base = AutoModelForMultimodalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16, device_map="cpu",
                                                low_cpu_mem_usage=True)
merged = PeftModel.from_pretrained(base, f"{OUT}/adapter").merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True)
AutoProcessor.from_pretrained(BASE_MODEL).save_pretrained(MERGED)
del base, merged; gc.collect()
print("merged model in", MERGED)

## 8. Convert to GGUF and quantize to Q4_K_M

Same quantization as the PC's untrained `gemma4:12b`, so before/after comparisons are fair.
Needs a llama.cpp recent enough to convert Gemma 4; if the convert step errors on the
architecture, stop here and keep the adapter (step 6).

In [ ]:
GGUF_BF16 = "/content/sno-advisor-bf16.gguf"
GGUF_Q4 = f"{OUT}/sno-advisor-{RUN}.Q4_K_M.gguf"
if not os.path.exists("/content/llama.cpp"):
    !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED} --outtype bf16 --outfile {GGUF_BF16}
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize {GGUF_BF16} {GGUF_Q4} Q4_K_M
print("saved", GGUF_Q4)

## 9. Back on the PC

Download `sno-advisor-r0.Q4_K_M.gguf` from Drive into a folder (not the repo), and put
this `Modelfile` next to it. `RENDERER`/`PARSER` are what the PC's `gemma4:12b` uses,
so the prompt is built the same way as in training:

```
FROM ./sno-advisor-r0.Q4_K_M.gguf
RENDERER gemma4
PARSER gemma4
PARAMETER temperature 0
```

Then:

```
ollama create sno-advisor:r0 -f Modelfile
ollama run sno-advisor:r0 --think=false "hello"
```

Next in the repo: `advice run sno-advisor:r0` on the 60 held-out stories, and the
comparison with untrained Gemma and Claude.